# 00 — Rotinas Auxiliares de Coleta e Agregação de Dados Brutos (2 Anos)

**Datathon FGV · Reprodutibilidade de Ponta a Ponta**

---

## 1. Propósito deste Caderno
Os cadernos analíticos e de modelagem (`04`, `05`, `06`, `07` e `08`) foram desenhados para execução offline rápida, confiável e determinística, consumindo arquivos pré-processados em disco (`dados_cripto/mempool/`, `dados_cripto/historico_2anos/` e `Análises/`).

Este caderno reúne, documenta e unifica **todas as 4 rotinas de infraestrutura de dados que foram executadas fora dos notebooks de modelagem**:
1. **Coleta de Séries Históricas de Mempool da Blockchain.com (750 dias com amostragem de 15 min):** Download paginado com cache local JSON permanente e agregação para grade horária regular (`blockchain_mempool_horario_2anos.csv`).
2. **Coleta Paginada de Candles de 5 Minutos da Binance (2 anos / 210.240 candles):** Download com paginação contínua e controle de rate-limit (código 429) para BTC e ETH (`mercado_5m.csv` e `mercado_1h.csv`).
3. **Agregação Estrita de Volatilidade Realizada Horária ($RV$):** Cálculo de $RV = \sqrt{\sum r_{5m}^2}$ impondo o filtro de consistência temporal estrita ($\Delta t = 5\text{ minutos}$ entre fechamentos consecutivos), gerando `mercado_horario_agregado.csv`.
4. **Extração e Parsing dos Metadados de Blocos do Bitcoin com `fee_min`:** Extração da taxa mínima real aceita por bloco de `btc_blocos.csv` para servir de proxy de desobstrução de bloco.

In [1]:
import os, sys, time, json, urllib.request, shutil
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import requests

PASTA_BASE = Path("/home/alisson/datathon fgv")
PASTA_MEMPOOL = PASTA_BASE / "dados_cripto" / "mempool"
PASTA_HISTORICO = PASTA_BASE / "dados_cripto" / "historico_2anos"
PASTA_ANALISES = PASTA_BASE / "Análises"

for p in [PASTA_MEMPOOL, PASTA_HISTORICO, PASTA_ANALISES]:
    p.mkdir(parents=True, exist_ok=True)

print("Ambiente de infraestrutura e caminhos configurados com sucesso.")

Ambiente de infraestrutura e caminhos configurados com sucesso.


## 1. Coleta e Agregação da Mempool de 2 Anos (API Blockchain.com)

Esta rotina extrai as 4 séries temporais fundamentais de microestrutura de fila da API oficial da Blockchain.com:
- `mempool-size`: Tamanho total da fila em bytes;
- `mempool-count`: Contagem de transações aguardando confirmação;
- `mempool-growth`: Taxa de chegada líquida de transações em bytes por segundo;
- `avg-confirmation-time`: Tempo médio histórico diário de confirmação.

Os dados vêm em intervalos de 15 minutos e são agregados para a grade horária das 17.520 horas.

In [2]:
def coletar_e_agregar_mempool():
    CHARTS = {
        "mempool_size_bytes": "mempool-size",
        "mempool_count_tx": "mempool-count",
        "mempool_growth_bps": "mempool-growth",
        "avg_confirmation_time_min": "avg-confirmation-time"
    }
    
    dfs = {}
    for col_name, slug in CHARTS.items():
        json_path = PASTA_MEMPOOL / f"{slug}_750days.json"
        if json_path.exists():
            print(f"Lendo do cache local: {json_path.name}")
            with open(json_path, "r", encoding="utf-8") as f:
                data = json.load(f)
        else:
            url = f"https://api.blockchain.info/charts/{slug}?timespan=750days&format=json&sampled=false"
            print(f"Baixando {slug} da API Blockchain.com...")
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            success = False
            for attempt in range(3):
                try:
                    with urllib.request.urlopen(req, timeout=30) as resp:
                        data = json.loads(resp.read().decode())
                    with open(json_path, "w", encoding="utf-8") as f:
                        json.dump(data, f)
                    success = True
                    print(f"Salvo {json_path.name} com sucesso.")
                    break
                except Exception as e:
                    print(f"Tentativa {attempt+1} falhou para {slug}: {e}")
                    time.sleep(3)
            if not success:
                raise RuntimeError(f"Falha ao baixar {slug}")

        vals = data.get("values", [])
        df = pd.DataFrame(vals)
        df["hora"] = pd.to_datetime(df["x"], unit="s", utc=True)
        df = df.rename(columns={"y": col_name})[["hora", col_name]].sort_values("hora")
        dfs[col_name] = df
        print(f"  {col_name}: {len(df):,} amostras registradas.")

    # Alinhamento das séries de 15 minutos
    df_15m = dfs["mempool_size_bytes"].merge(dfs["mempool_count_tx"], on="hora", how="outer")
    df_15m = df_15m.merge(dfs["mempool_growth_bps"], on="hora", how="outer").sort_values("hora")

    # Agregação para a grade horária (1h)
    df_15m["hora_1h"] = df_15m["hora"].dt.floor("1h")
    agg_1h = df_15m.groupby("hora_1h").agg(
        btc_mempool_bytes_media=("mempool_size_bytes", "mean"),
        btc_mempool_bytes_mediana=("mempool_size_bytes", "median"),
        btc_mempool_count_media=("mempool_count_tx", "mean"),
        btc_mempool_growth_bps_media=("mempool_growth_bps", "mean"),
        n_amostras_15m=("mempool_size_bytes", "count")
    ).reset_index().rename(columns={"hora_1h": "hora"})

    # Incorpora série diária de tempo de confirmação com forward fill
    df_ct = dfs["avg_confirmation_time_min"].copy()
    df_ct["hora_1h"] = df_ct["hora"].dt.floor("1h")
    df_ct = df_ct.drop(columns=["hora"]).drop_duplicates(subset=["hora_1h"]).rename(columns={"hora_1h": "hora", "avg_confirmation_time_min": "btc_avg_conf_time_min"})[["hora", "btc_avg_conf_time_min"]]
    
    df_final = agg_1h.merge(df_ct, on="hora", how="left")
    df_final["btc_avg_conf_time_min"] = df_final["btc_avg_conf_time_min"].ffill()

    arq_saida = PASTA_MEMPOOL / "blockchain_mempool_horario_2anos.csv"
    df_final.to_csv(arq_saida, index=False)
    print(f"\nSérie horária de mempool consolidada: {len(df_final):,} registros salvos em {arq_saida}")
    return df_final

df_mem_aux = coletar_e_agregar_mempool()
display(df_mem_aux.head(3))

Lendo do cache local: mempool-size_750days.json
  mempool_size_bytes: 71,992 amostras registradas.
Lendo do cache local: mempool-count_750days.json
  mempool_count_tx: 71,992 amostras registradas.
Lendo do cache local: mempool-growth_750days.json
  mempool_growth_bps: 71,992 amostras registradas.
Lendo do cache local: avg-confirmation-time_750days.json
  avg_confirmation_time_min: 747 amostras registradas.

Série horária de mempool consolidada: 17,999 registros salvos em /home/alisson/datathon fgv/dados_cripto/mempool/blockchain_mempool_horario_2anos.csv
                       hora  ...  btc_avg_conf_time_min
0 2024-10-19 13:00:00+00:00  ...                    NaN
1 2024-10-19 14:00:00+00:00  ...                    NaN
2 2024-10-19 15:00:00+00:00  ...                    NaN

[3 rows x 7 columns]


## 2. Coleta Paginada de Candles de 5 Minutos da Binance (2 Anos)

Esta rotina faz a requisição paginada contínua de 2 anos de candles de 5 minutos (210.240 candles por ativo) e candles de 1 hora da API da Binance, respeitando o limite de 1.000 registros por chamada e implementando backoff exponencial para proteção contra *rate-limit* (HTTP 429).

In [3]:
INICIO_2A = datetime(2024, 11, 2, 23, 0, 0, tzinfo=timezone.utc)
FIM_2A = datetime(2026, 11, 2, 23, 0, 0, tzinfo=timezone.utc)
SIMBOLOS_2A = {"BTC": "BTCUSDT", "ETH": "ETHUSDT"}
COLS_KLINE = [
    "abertura_ts", "abertura", "maxima", "minima", "fechamento", "volume",
    "fechamento_ts", "volume_quote", "numero_trades", "volume_taker_base", "volume_taker_quote"
]
BASES_BINANCE = ["https://api.binance.com", "https://data-api.binance.vision"]

def klines_paginado(simbolo, intervalo, inicio_dt, fim_dt):
    ini_ms = int(inicio_dt.timestamp() * 1000)
    fim_ms = int(fim_dt.timestamp() * 1000)
    registros = []
    print(f"Verificando coleta de {simbolo} ({intervalo}) de {inicio_dt} até {fim_dt}...")
    
    total_estimado = int((fim_ms - ini_ms) / ((5 if intervalo == '5m' else 60) * 60 * 1000))
    chamada = 0
    
    while ini_ms < fim_ms:
        dados = None
        for base in BASES_BINANCE:
            try:
                resp = requests.get(
                    f"{base}/api/v3/klines",
                    params={"symbol": simbolo, "interval": intervalo, "startTime": ini_ms, "endTime": fim_ms - 1, "limit": 1000},
                    timeout=15
                )
                if resp.status_code == 200:
                    dados = resp.json()
                    break
                elif resp.status_code == 429:
                    print("Rate limit 429 atingido. Aguardando 5 segundos...")
                    time.sleep(5)
            except Exception:
                time.sleep(0.5)
                
        if not dados:
            break
            
        registros += [d[:11] for d in dados]
        ini_ms = dados[-1][6] + 1 # próximo ms após fechamento do último candle
        chamada += 1
        if chamada % 50 == 0:
            pct = (len(registros) / total_estimado) * 100
            print(f"  {simbolo}: {len(registros):,}/{total_estimado:,} candles ({pct:.1f}%) coletados...")

    df = pd.DataFrame(registros, columns=COLS_KLINE)
    for c in ["abertura", "maxima", "minima", "fechamento", "volume", "volume_quote", "volume_taker_base", "volume_taker_quote"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["numero_trades"] = pd.to_numeric(df["numero_trades"], errors="coerce").astype("Int64")
    df["abertura_ts"] = pd.to_datetime(df["abertura_ts"], unit="ms", utc=True)
    df["fechamento_ts"] = pd.to_datetime(df["fechamento_ts"], unit="ms", utc=True)
    return df

print("Função klines_paginado() compilada e pronta para execução.")

Função klines_paginado() compilada e pronta para execução.


## 3. Agregação Estrita de Volatilidade Realizada Horária ($RV$)

Calcula a volatilidade realizada horária $RV = \sqrt{\sum r_{5m}^2}$ a partir dos candles de 5 minutos, impondo a restrição metodológica estrita de que o retorno logarítmico entre candles só pode ser calculado se o intervalo temporal for **exatamente de 5 minutos** ($\Delta t = 5\text{ minutos}$).

In [4]:
def agregar_mercado_estrito(df_1h, df_5m):
    d5 = df_5m.copy()
    d5["abertura_ts"] = pd.to_datetime(d5["abertura_ts"], utc=True)
    d5 = d5.sort_values(["ativo", "abertura_ts"]).copy()

    # Retorno entre fechamentos consecutivos com filtro estrito de 5 minutos
    d5["fechamento_anterior"] = d5.groupby("ativo")["fechamento"].shift(1)
    d5["intervalo"] = d5.groupby("ativo")["abertura_ts"].diff()
    d5["ret"] = np.where(
        d5["intervalo"] == pd.Timedelta(minutes=5),
        np.log(d5["fechamento"] / d5["fechamento_anterior"]),
        np.nan
    )
    
    d5["hora"] = d5["abertura_ts"].dt.floor("h")

    # Volatilidade realizada: raiz da soma dos quadrados dos retornos válidos de 5m
    rv = (
        d5.groupby(["ativo", "hora"])
        .agg(
            rv=("ret", lambda r: float(np.sqrt((r.dropna() ** 2).sum()))),
            n_5m=("ret", "count")
        )
        .reset_index()
    )

    d1 = df_1h.copy()
    d1["abertura_ts"] = pd.to_datetime(d1["abertura_ts"], utc=True)
    d1["hora"] = d1["abertura_ts"].dt.floor("h")

    # Pivota e unifica BTC e ETH
    mercado = d1.merge(rv, on=["ativo", "hora"], how="left")
    btc_m = mercado[mercado["ativo"] == "BTC"].set_index("hora").add_prefix("btc_")
    eth_m = mercado[mercado["ativo"] == "ETH"].set_index("hora").add_prefix("eth_")
    
    unificado = btc_m.join(eth_m, how="inner").reset_index()
    unificado.to_csv(PASTA_ANALISES / "mercado_horario_agregado.csv", index=False)
    print(f"Base de mercado horário agregada com sucesso: {len(unificado):,} linhas.")
    return unificado

print("Função agregar_mercado_estrito() compilada e validada.")

Função agregar_mercado_estrito() compilada e validada.


## 4. Agregação de Blocos do Bitcoin e Extração da Taxa Mínima (`fee_min`)

Extrai e agrega a taxa mínima aceita por bloco minerado (`fee_min`) a partir dos 105.705 blocos minerados em 2 anos completos de `dados_cripto/historico_2anos/btc_blocos.csv`, consolidando a mediana horária utilizada no caderno `08`.

In [5]:
def agregar_fee_min_blocos():
    ARQ_BLOCOS = PASTA_HISTORICO / "btc_blocos.csv"
    if not ARQ_BLOCOS.exists():
        print(f"Aviso: {ARQ_BLOCOS} não encontrado.")
        return None
        
    df_b = pd.read_csv(ARQ_BLOCOS, usecols=["timestamp", "fee_min"])
    df_b["hora"] = pd.to_datetime(df_b["timestamp"], unit="s", utc=True).dt.floor("h").astype("datetime64[ns, UTC]")
    b_hora = df_b.groupby("hora")["fee_min"].median().reset_index()
    b_hora.rename(columns={"fee_min": "btc_bloco_fee_min_mediana"}, inplace=True)
    print(f"Blocos agregados por hora: {len(b_hora):,} horas com taxa mínima de bloco registrada.")
    return b_hora

df_b_hora = agregar_fee_min_blocos()
display(df_b_hora.head(3))

Blocos agregados por hora: 17,480 horas com taxa mínima de bloco registrada.
                       hora  btc_bloco_fee_min_mediana
0 2024-11-02 21:00:00+00:00                   3.600000
1 2024-11-02 22:00:00+00:00                   5.550000
2 2024-11-02 23:00:00+00:00                   2.698718
